In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import keras
from keras import layers

path = keras.utils.get_file("kalevala.txt",
                            "https://www.gutenberg.org/cache/epub/7000/pg7000.txt")
raw = open(path, encoding="utf-8").read()
poems = raw[raw.index("Ensimmäinen runo"):raw.index("*** END OF")]
lines = [line.strip() for line in poems.splitlines() if line.strip()]
text = " / ".join(lines)
print(len(lines), "säettä,", len(text), "merkkiä")
print(text[:100])

I0000 00:00:1793805366.500614  243871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793805366.533045  243871 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793805367.188514  243871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


22846 säettä, 591030 merkkiä
Ensimmäinen runo / Mieleni minun tekevi, / aivoni ajattelevi / lähteäni laulamahan, / saa'ani sanele


In [2]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Metaspace()
tokenizer.decoder = decoders.Metaspace()
trainer = trainers.BpeTrainer(vocab_size=4000, special_tokens=["[UNK]"],
                              show_progress=False)
tokenizer.train_from_iterator([text], trainer)

encoding = tokenizer.encode("Vaka vanha Väinämöinen")
print(encoding.tokens)
print(encoding.ids)

['▁Vaka', '▁vanha', '▁Väinämöinen']
[565, 241, 289]


In [3]:
ids = np.array(tokenizer.encode(text).ids)
print(len(ids), "tokenia")
seq_length = 64
split = int(0.9 * len(ids))

def windows(tokens, step):
    starts = range(0, len(tokens) - seq_length - 1, step)
    x = np.stack([tokens[s:s + seq_length] for s in starts])
    y = np.stack([tokens[s + 1:s + seq_length + 1] for s in starts])
    return x, y

x_train, y_train = windows(ids[:split], step=16)
x_val, y_val = windows(ids[split:], step=seq_length)
print(x_train.shape, x_val.shape)

156438 tokenia
(8796, 64) (244, 64)


In [4]:
def positional_encoding(length, dim):
    positions = np.arange(length)[:, None]
    rates = 1 / 10000 ** (np.arange(dim)[None, :] / dim)
    angles = positions * rates
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return angles.astype("float32")

def transformer_block(x, dim, heads, ff_dim):
    a = layers.MultiHeadAttention(num_heads=heads, key_dim=dim // heads)(
        x, x, use_causal_mask=True)
    x = layers.LayerNormalization()(x + a)
    f = layers.Dense(ff_dim, activation="relu")(x)
    f = layers.Dense(dim)(f)
    return layers.LayerNormalization()(x + f)

keras.utils.set_random_seed(0)
vocab_size = tokenizer.get_vocab_size()
dim, heads, ff_dim = 256, 4, 512
inputs = keras.Input(shape=(seq_length,), dtype="int32")
x = layers.Embedding(vocab_size, dim)(inputs)
x = x + positional_encoding(seq_length, dim)
for _ in range(2):
    x = transformer_block(x, dim, heads, ff_dim)
outputs = layers.Dense(vocab_size)(x)
model = keras.Model(inputs, outputs)
print(f"parametreja {model.count_params():,}")

I0000 00:00:1793805368.608147  243871 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1392 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9


parametreja 3,106,208


In [5]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
stop = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                    epochs=30, batch_size=32, callbacks=[stop], verbose=2)

Epoch 1/30


I0000 00:00:1793805370.690136  244064 service.cc:153] XLA service 0x7fb0f4045fb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1793805370.690151  244064 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Ti, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1793805370.739188  244064 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1793805370.886364  244064 assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1793805371.073461  244064 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1793805371.091545  244064 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_7161__.83
I0000 00:00:1793805371.463944  244185 subprocess_compilation.cc:348] ptxas warning : Registe

275/275 - 27s - 98ms/step - accuracy: 0.1455 - loss: 7.0131 - val_accuracy: 0.1458 - val_loss: 6.7493
Epoch 2/30
275/275 - 1s - 4ms/step - accuracy: 0.1536 - loss: 6.1979 - val_accuracy: 0.1559 - val_loss: 6.0435
Epoch 3/30
275/275 - 1s - 4ms/step - accuracy: 0.1623 - loss: 5.7142 - val_accuracy: 0.1627 - val_loss: 5.8116
Epoch 4/30
275/275 - 1s - 4ms/step - accuracy: 0.1701 - loss: 5.4415 - val_accuracy: 0.1735 - val_loss: 5.6921
Epoch 5/30
275/275 - 1s - 4ms/step - accuracy: 0.1810 - loss: 5.1905 - val_accuracy: 0.1797 - val_loss: 5.6147
Epoch 6/30
275/275 - 1s - 4ms/step - accuracy: 0.1992 - loss: 4.8643 - val_accuracy: 0.1838 - val_loss: 5.6138
Epoch 7/30
275/275 - 1s - 4ms/step - accuracy: 0.2261 - loss: 4.4945 - val_accuracy: 0.1908 - val_loss: 5.6435
Epoch 8/30
275/275 - 1s - 4ms/step - accuracy: 0.2570 - loss: 4.1363 - val_accuracy: 0.1936 - val_loss: 5.7755


In [6]:
def generate(prompt, tokens=50, temperature=1.0):
    ids = tokenizer.encode(prompt).ids
    for _ in range(tokens):
        context = ids[-seq_length:]
        x = np.zeros((1, seq_length), dtype="int32")
        x[0, :len(context)] = context
        logits = model.predict(x, verbose=0)[0, len(context) - 1]
        p = np.exp((logits - logits.max()) / temperature)
        p /= p.sum()
        ids.append(int(np.random.choice(len(p), p=p)))
    return tokenizer.decode(ids).replace(" / ", "\n")

np.random.seed(1)
for temperature in [0.5, 1.0, 1.5]:
    print(f"--- T = {temperature}")
    print(generate("Vaka vanha Väinämöinen", temperature=temperature))

--- T = 0.5


I0000 00:00:1793805403.714040  244061 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_18209__.13
I0000 00:00:1793805403.788494  244061 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1793805404.029370  245934 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_13', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1793805404.208348  244061 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1793805404.520445  245923 subprocess_compilation.cc:348] ptxas warning : Registers are spilled t

Vaka vanha Väinämöinen
saigetpää iltelevi,
luikui ripikutelevi,
tapes'inki rinnalainen.
Sys' onpä työnnun tulevähän
näillä laastajäksi,
ilman keittivikkollä; /
--- T = 1.0
Vaka vanha Väinämöinen
ajoi siitä suuri lintu:
ei varsin ammin maaten,
latahan paimenllä nälkän,
karvasti vaskisahan."
Virkkoi nuori Lemminkäinen!
takoja iän-ikuinen,
otavaisensa: noin nimesi:
vierähti valmja,
et kurjat, är
--- T = 1.5
Vaka vanha Väinämöinen
siir'on kullan veräjän haunänsä.
"Itja Väinämöinen /ehe miä'itä, tukka kuollehtä,
kukäh Ilmarin seläistä, puahellasi, rekehen,
jätti täuli saanen.
Virkkoiytpä mon haehen.
S
